# Welcome to your first assignment!

Instructions are below. Please give this a try, and look in the solutions folder if you get stuck (or feel free to ask me!)

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Just before we get to the assignment --</h2>
            <span style="color:#f71;">I thought I'd take a second to point you at this page of useful resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
And in another Terminal (Mac) or Powershell (Windows), enter `ollama pull llama3.2`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

If Ollama is slow on your machine, try using `llama3.2:1b` as an alternative. Run `ollama pull llama3.2:1b` from a Terminal or Powershell, and change the code below from `MODEL = "llama3.2"` to `MODEL = "llama3.2:1b"`

In [1]:
# imports

import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display

In [2]:
# Constants

OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {"Content-Type": "application/json"}
MODEL = "llama3.2"

In [3]:
# Create a messages list using the same format that we used for OpenAI

messages = [
    {"role": "user", "content": "Describe some of the business applications of Generative AI"}
]

In [4]:
payload = {
        "model": MODEL,
        "messages": messages,
        "stream": False
    }

In [ ]:
# Let's just make sure the model is loaded
# Ignore this.. we run it ollama in a container.. 
# you should first connect to the container to make this work
#  docker container exec -it ollama bash
#  ollama pull llama3.2
#   
!ollama pull llama3.2

/bin/bash: line 1: ollama: command not found


In [9]:
# If this doesn't work for any reason, try the 2 versions in the following cells
# And double check the instructions in the 'Recap on installation of Ollama' at the top of this lab
# And if none of that works - contact me!

response = requests.post(OLLAMA_API, json=payload, headers=HEADERS)
print(response.json()['message']['content'])

Generative AI (Artificial Intelligence) has numerous business applications across various industries. Here are some examples:

1. **Content Generation**: Generative AI can be used to generate high-quality content, such as:
 * Articles and blog posts
 * Social media posts and comments
 * Product descriptions and sales copy
 * Music and audio tracks
2. **Design and Visualization**: Generative AI can assist in design and visualization tasks, including:
 * Graphic design (e.g., logos, icons, and infographics)
 * Architecture and interior design
 * Video game development
 * 3D modeling and animation
3. **Marketing and Advertising**: Generative AI can help with marketing and advertising efforts by:
 * Generating personalized ad copy and landing pages
 * Creating product recommendations based on user behavior
 * Developing chatbots for customer service and support
4. **Data Analysis and Visualization**: Generative AI can be used to analyze and visualize large datasets, including:
 * Predictiv

In [8]:
response.json()

{'error': 'model "llama3.2" not found, try pulling it first'}

# Introducing the ollama package

And now we'll do the same thing, but using the elegant ollama python package instead of a direct HTTP call.

Under the hood, it's making the same call as above to the ollama server running at localhost:11434

In [10]:
import ollama

response = ollama.chat(model=MODEL, messages=messages)
print(response['message']['content'])

Generative AI has numerous business applications across various industries, including:

1. **Content Creation**: AI-powered tools can generate high-quality content such as blog posts, articles, social media posts, and product descriptions, reducing the need for human writers and editors.
2. **Marketing Automation**: Generative AI can create personalized marketing messages, product recommendations, and customer personas to enhance marketing campaigns and improve ROI.
3. **Data Analysis and Visualization**: AI-powered tools can analyze large datasets, identify patterns, and generate visualizations to help businesses gain insights into customer behavior, market trends, and operational efficiency.
4. **Customer Service and Support**: Chatbots powered by Generative AI can provide 24/7 customer support, responding to queries, resolving issues, and routing complex problems to human representatives.
5. **Product Design and Development**: Generative AI tools can create 3D models, product protot

## Alternative approach - using OpenAI python library to connect to Ollama

In [11]:
# There's actually an alternative approach that some people might prefer
# You can use the OpenAI client python library to call Ollama:

from openai import OpenAI
ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

response = ollama_via_openai.chat.completions.create(
    model=MODEL,
    messages=messages
)

print(response.choices[0].message.content)

Generative AI has numerous potential business applications across various industries. Here are some examples:

1. **Content Generation**: Generative AI can be used to generate high-quality content such as:
	* Articles and reports on demand
	* Social media posts and tweets
	* Product descriptions and reviews
	* Marketing copy and slogans
2. **Image and Video Creation**: Generative AI can create realistic images and videos for:
	* Advertising and marketing campaigns
	* Entertainment industry (e.g., movie and TV show thumbnails)
	* Fashion and beauty brands (e.g., product illustrations and showcases)
3. **Data Analysis and Insights**: Generative AI can help analyze large datasets to identify patterns, trends, and correlations, providing valuable insights for:
	* Market research and competitor analysis
	* Customer behavior and behavior modeling
	* Predictive analytics and forecast generation
4. **Chatbots and Virtual Assistants**: Generative AI can be used to create more sophisticated chat

## Are you confused about why that works?

It seems strange, right? We just used OpenAI code to call Ollama?? What's going on?!

Here's the scoop:

The python class `OpenAI` is simply code written by OpenAI engineers that makes calls over the internet to an endpoint.  

When you call `openai.chat.completions.create()`, this python code just makes a web request to the following url: "https://api.openai.com/v1/chat/completions"

Code like this is known as a "client library" - it's just wrapper code that runs on your machine to make web requests. The actual power of GPT is running on OpenAI's cloud behind this API, not on your computer!

OpenAI was so popular, that lots of other AI providers provided identical web endpoints, so you could use the same approach.

So Ollama has an endpoint running on your local box at http://localhost:11434/v1/chat/completions  
And in week 2 we'll discover that lots of other providers do this too, including Gemini and DeepSeek.

And then the team at OpenAI had a great idea: they can extend their client library so you can specify a different 'base url', and use their library to call any compatible API.

That's it!

So when you say: `ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')`  
Then this will make the same endpoint calls, but to Ollama instead of OpenAI.

## Also trying the amazing reasoning model DeepSeek

Here we use the version of DeepSeek-reasoner that's been distilled to 1.5B.  
This is actually a 1.5B variant of Qwen that has been fine-tuned using synethic data generated by Deepseek R1.

Other sizes of DeepSeek are [here](https://ollama.com/library/deepseek-r1) all the way up to the full 671B parameter version, which would use up 404GB of your drive and is far too large for most!

In [ ]:
!ollama pull deepseek-r1:1.5b

In [ ]:
# This may take a few minutes to run! You should then see a fascinating "thinking" trace inside <think> tags, followed by some decent definitions

response = ollama_via_openai.chat.completions.create(
    model="deepseek-r1:1.5b",
    messages=[{"role": "user", "content": "Please give definitions of some core concepts behind LLMs: a neural network, attention and the transformer"}]
)

print(response.choices[0].message.content)

# NOW the exercise for you

Take the code from day1 and incorporate it here, to build a website summarizer that uses Llama 3.2 running locally instead of OpenAI; use either of the above approaches.

In [14]:
import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import ollama

headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


class Website:

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)


system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."


def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt


def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

def summarize(url):
    website = Website(url)
    response = ollama.chat(model="llama3.2", 
                           messages= messages_for(website)
    )
    return response['message']['content']


def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))


In [ ]:
w = Website("https://longwave.it")
m = messages_for(w)


[{'role': 'system', 'content': 'You are an assistant that analyzes the contents of a website and provides a short summary, ignoring text that might be navigation related. Respond in markdown.'}, {'role': 'user', 'content': "You are looking at a website titled Longwave | Your Digital Advisor\nThe contents of this website is as follows; please provide a short summary of this website in markdown. If it includes news or announcements, then summarize these too.\n\nPer usufruire di tutte le funzionalità di questo sito è necessario\nabilitare Javascript\n. Qui puoi trovare le\nistruzioni su come abilitare JavaScript nel tuo browser\n.\nContatti\nCustomer portal\nArea riservata\nLingue\nen\nit\nChi siamo\nL’etica del fare impresa\nManifesto\nPartner\nMercati\nGrandi Imprese\nPubblica Amministrazione\nSanità\nScuole e Università\nPiccole e Medie Imprese\nTelco – Operatori Rete Fissa e Mobile\nMedia Broadcaster\nInternet Service Provider\nSoluzioni\nCybersecurity\nNetworking\nIT & Cloud\nUC & Au

In [24]:
print(m[0]['role'])
print(m[0]['content'])
print("------------")
print(m[1]['role'])
print(m[1]['content'])


system
You are an assistant that analyzes the contents of a website and provides a short summary, ignoring text that might be navigation related. Respond in markdown.
------------
user
You are looking at a website titled Longwave | Your Digital Advisor
The contents of this website is as follows; please provide a short summary of this website in markdown. If it includes news or announcements, then summarize these too.

Per usufruire di tutte le funzionalità di questo sito è necessario
abilitare Javascript
. Qui puoi trovare le
istruzioni su come abilitare JavaScript nel tuo browser
.
Contatti
Customer portal
Area riservata
Lingue
en
it
Chi siamo
L’etica del fare impresa
Manifesto
Partner
Mercati
Grandi Imprese
Pubblica Amministrazione
Sanità
Scuole e Università
Piccole e Medie Imprese
Telco – Operatori Rete Fissa e Mobile
Media Broadcaster
Internet Service Provider
Soluzioni
Cybersecurity
Networking
IT & Cloud
UC & Audio-Video Integration
Modern Workplace
Software – Mercury Suite
Serviz

In [26]:
print("Using: " + OLLAMA_API)
display_summary("https://longwave.it")

Usinng: http://localhost:11434/api/chat


### Longwave | Your Digital Advisor

#### Mission Statement and Values
Longwave is a digital advisor that provides comprehensive, structured, and certified consulting to reduce the perceived complexity of technology.

#### Services
The website outlines several services offered by Longwave, including:

*   Cybersecurity
*   Networking
*   IT & Cloud
*   UC & Audio-Video Integration
*   Modern Workplace
*   Software – Mercury Suite
*   Servizi
    *   Cybersecurity Services
    *   LION\*
    *   Care4You
    *   RED Team

#### Solution Approach
Longwave takes a bespoke approach to digital technology, creating tailored solutions for each client's objectives.

#### Success Stories
The website showcases several business cases of success for various clients across different industries, including:

*   Meeting Rooms & Auditorium for an energy company
*   Medical Device protection and security services for ASL Alessandria
*   Digital Event Management for Bonatti

#### Events
Longwave participates in various events, including the Fortinet Security Days 2025.

#### Team and Culture
The website highlights Longwave's commitment to innovation and talent recruitment.